In [1]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))

from src.train import load_xy, split_train_test

X, y = load_xy()
X_train, X_test, y_train, y_test = split_train_test(X, y)
print(f"train {len(X_train)} mẫu | test {len(X_test)} mẫu "
      f"(test giữ nguyên, chưa từng được model nhìn thấy)")


train 320 mẫu | test 80 mẫu (test giữ nguyên, chưa từng được model nhìn thấy)


In [2]:
from src.train import run_baseline

baseline = run_baseline(X_train, y_train)
baseline[["model", "recall_ckd_mean", "recall_ckd_std",
          "precision_mean", "f1_mean", "roc_auc_mean"]]



[8.1] Chạy baseline (5 mô hình, coding=ordinal, 15 fold)...


        - dummy   recall_ckd=1.000 ± 0.000  (1.2s)


        - logreg  recall_ckd=0.987 ± 0.015  (1.3s)


        - dt      recall_ckd=0.967 ± 0.020  (1.2s)


        - rf      recall_ckd=0.997 ± 0.008  (5.6s)


        - knn     recall_ckd=0.935 ± 0.031  (1.6s)
        → đã lưu D:\New folder\kidney-disease-project\reports\tables\cv_baseline.csv


,model,recall_ckd_mean,recall_ckd_std,precision_mean,f1_mean,roc_auc_mean
0,dummy,1.000000,0.000000,0.625000,0.769231,0.500000
3,rf,0.996667,0.008498,0.990203,0.993374,0.999653
1,logreg,0.986667,0.015456,1.000000,0.993227,1.000000
2,dt,0.966667,0.019720,0.978368,0.972294,0.965278
4,knn,0.935000,0.031358,1.000000,0.966136,0.994236


In [3]:
from src.train import run_grid_search

results = run_grid_search(X_train, y_train)
results.sort_values("recall_ckd_mean", ascending=False)[
    ["model", "coding", "recall_ckd_mean", "recall_ckd_std",
     "f1_mean", "roc_auc_mean", "best_params"]]



[8.2] GridSearchCV: 4 mô hình × 2 kiểu mã hóa sg/al/su ...


        - logreg | ordinal | recall=0.992 ± 0.020 | f1=0.994 | auc=1.000 | 8.5s


        - logreg | onehot  | recall=0.992 ± 0.012 | f1=0.995 | auc=1.000 | 1.6s


        - dt     | ordinal | recall=0.985 ± 0.015 | f1=0.978 | auc=0.976 | 1.6s


        - dt     | onehot  | recall=0.962 ± 0.022 | f1=0.966 | auc=0.961 | 1.5s


        - rf     | ordinal | recall=0.997 ± 0.008 | f1=0.993 | auc=1.000 | 6.1s


        - rf     | onehot  | recall=0.992 ± 0.012 | f1=0.991 | auc=0.999 | 6.0s


        - knn    | ordinal | recall=0.945 ± 0.040 | f1=0.971 | auc=0.986 | 4.0s


        - knn    | onehot  | recall=0.925 ± 0.037 | f1=0.961 | auc=0.983 | 3.8s
        → đã lưu D:\New folder\kidney-disease-project\reports\tables\cv_results.csv


,model,coding,recall_ckd_mean,recall_ckd_std,f1_mean,roc_auc_mean,best_params
4,rf,ordinal,0.996667,0.008498,0.993374,0.999653,"{""clf__max_depth"": null, ""clf__n_estimators"": ..."
0,logreg,ordinal,0.991667,0.019720,0.994089,1.000000,"{""clf__C"": 10}"
5,rf,onehot,0.991667,0.011785,0.990843,0.999375,"{""clf__max_depth"": null, ""clf__n_estimators"": ..."
1,logreg,onehot,0.991667,0.011785,0.994958,0.999931,"{""clf__C"": 10}"
2,dt,ordinal,0.985000,0.015275,0.978496,0.975868,"{""clf__max_depth"": 3}"
3,dt,onehot,0.961667,0.022111,0.966452,0.960799,"{""clf__max_depth"": 3}"
6,knn,ordinal,0.945000,0.040000,0.971280,0.986042,"{""clf__n_neighbors"": 3, ""clf__weights"": ""unifo..."
7,knn,onehot,0.925000,0.036515,0.960667,0.983160,"{""clf__n_neighbors"": 3, ""clf__weights"": ""unifo..."


In [4]:
from src.train import choose_winner

winner = choose_winner(results)



[8.3] Chọn mô hình theo quy tắc: recall cao nhất, chênh < 1 độ lệch chuẩn thì ưu tiên đơn giản:
 model  coding  recall_ckd_mean  recall_ckd_std  f1_mean  roc_auc_mean                                        best_params
    dt ordinal         0.985000        0.015275 0.978496      0.975868                              {"clf__max_depth": 3}
   knn ordinal         0.945000        0.040000 0.971280      0.986042 {"clf__n_neighbors": 3, "clf__weights": "uniform"}
logreg ordinal         0.991667        0.019720 0.994089      1.000000                                     {"clf__C": 10}
    rf ordinal         0.996667        0.008498 0.993374      0.999653 {"clf__max_depth": null, "clf__n_estimators": 200}
        → THẮNG: logreg (coding=ordinal, recall=0.992 ± 0.020)


In [5]:
from src.train import fit_winner_on_full_train, save_artifacts

pipeline, search = fit_winner_on_full_train(X_train, y_train, winner)
save_artifacts(pipeline, winner, search, n_train=len(X_train), n_test=len(X_test))



[8.4] Fit model cuối trên toàn bộ tập train ...


        siêu tham số thắng: {"clf__C": 10}

        → đã lưu D:\New folder\kidney-disease-project\models\pipeline.joblib
        → đã lưu D:\New folder\kidney-disease-project\models\metadata.json


In [6]:
from src.evaluate import (
    load_model_and_test, compute_metrics,
    plot_confusion_matrix, plot_roc_curve,
)
from src.config import FIGURES_DIR
from sklearn.metrics import confusion_matrix

model, metadata, X_eval, y_eval = load_model_and_test()
y_pred = model.predict(X_eval)
y_proba = model.predict_proba(X_eval)[:, 1]

metrics = compute_metrics(y_eval, y_pred, y_proba)
display(metrics)

print(f"CV (train, 15 fold): recall = {metadata['cv_recall_ckd_mean']:.3f} "
      f"± {metadata['cv_recall_ckd_std']:.3f}")
test_recall = float(metrics.loc[metrics.metric == "recall_ckd", "value"].iloc[0])
print(f"Test (80 mẫu)      : recall = {test_recall:.3f}")


,metric,value
0,accuracy,0.9750
1,precision_ckd,1.0000
2,recall_ckd,0.9600
3,f1_ckd,0.9796
4,roc_auc,0.9993


CV (train, 15 fold): recall = 0.992 ± 0.020
Test (80 mẫu)      : recall = 0.960


In [7]:
cm = confusion_matrix(y_eval, y_pred, labels=[0, 1])
print(f"TN={cm[0, 0]}  FP={cm[0, 1]}  FN={cm[1, 0]}  TP={cm[1, 1]}")

auc_value = float(metrics.loc[metrics.metric == "roc_auc", "value"].iloc[0])
plot_confusion_matrix(y_eval, y_pred, FIGURES_DIR / "confusion_matrix.png")
plot_roc_curve(y_eval, y_proba, auc_value, FIGURES_DIR / "roc_curve.png")

from IPython.display import Image, display as show_image
show_image(filename=str(FIGURES_DIR / "confusion_matrix.png"))
show_image(filename=str(FIGURES_DIR / "roc_curve.png"))


TN=30  FP=0  FN=2  TP=48
        → đã lưu D:\New folder\kidney-disease-project\reports\figures\confusion_matrix.png


        → đã lưu D:\New folder\kidney-disease-project\reports\figures\roc_curve.png


In [8]:
from src.evaluate import export_predictions, export_feature_importance
from src.config import TABLES_DIR

preds = export_predictions(X_eval, y_eval, y_pred, y_proba,
                           TABLES_DIR / "test_predictions.csv")
print("Danh sách lỗi (FN = bỏ sót CKD — đọc kỹ các dòng này):")
display(preds[preds["loại_lỗi"] != "đúng"])

export_feature_importance(pipeline, TABLES_DIR / "feature_importance.csv")


        → đã lưu D:\New folder\kidney-disease-project\reports\tables\test_predictions.csv
Danh sách lỗi (FN = bỏ sót CKD — đọc kỹ các dòng này):


,id,y_true,y_pred,prob_ckd,loại_lỗi
10,224,1,0,0.3875,FN — bỏ sót CKD
12,102,1,0,0.3065,FN — bỏ sót CKD


        → đã lưu D:\New folder\kidney-disease-project\reports\tables\feature_importance.csv (top 5: sg, hemo, pcv, sc, al)


,feature,importance
0,sg,2.9274
1,hemo,2.7863
2,pcv,2.5721
3,sc,2.1391
4,al,1.8923
5,htn_no,1.7228
6,htn_yes,1.7211
7,appet_good,1.7210
8,appet_poor,1.7193
9,bgr,1.5228
